# Exercise: Bias–Variance Tradeoff

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/AAI/blob/main/courses/Machine_Learning/exercises/06/01_bias-variance_tradeoff.ipynb)
- Data: **synthetic** — 30 points from a noisy quadratic (generated below)

---

A degree-15 polynomial can fit 30 training points almost perfectly — and still be useless on new data. **Why does more complexity sometimes make predictions worse?**

In this exercise you will:

- Recognize **under-fitting** (high bias) and **over-fitting** (high variance) from model curves and scores
- Estimate **generalization performance** with **cross-validation** instead of a single train/test split
- Compare models of different complexity using **cross-validated MAE**

**How this exercise works:**

- for filled code cells, pause and write down what you **expect** to see. Then run the cell and compare.
- some questions will require you to write code
- some questions will require you to write prose in markdown to answer them

![Bias–variance tradeoff](../../assets/bias-variance_tradeoff.png)


## Setup

In [ ]:
# https://pandas.pydata.org/docs/
import pandas as pd

# https://numpy.org/doc/
import numpy as np

# https://seaborn.pydata.org/
import seaborn as sns

# https://matplotlib.org/
import matplotlib.pyplot as plt

# https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
from sklearn.pipeline import Pipeline

# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html
from sklearn.preprocessing import PolynomialFeatures

# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
from sklearn.linear_model import LinearRegression

# https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html
from sklearn.dummy import DummyRegressor

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html
from sklearn.model_selection import cross_validate

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
from sklearn.model_selection import KFold


## Step 1 — Generate and visualize the data

We generate 30 points from a **quadratic** function plus noise. The true relationship is curved, so a straight line will underfit and a very high-degree polynomial may overfit.

The data-generation cell below is **provided** so everyone works with the same points.


In [ ]:
np.random.seed(11)

n_samples = 30
X = 6 * np.random.rand(n_samples, 1) - 3
y = 0.5 * X ** 2 + X + 2 + np.random.randn(n_samples, 1)

data_df = pd.DataFrame({"X": X.ravel(), "y": y.ravel()})
data_df.head()


### Step 1.b Plot the scatter

**Predict:** does the relationship look like a straight line or a curve?


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.scatterplot(
    data=data_df,
    x='X',
    y='y',
    ax=ax,
)
ax.set_title('Generated non-linear data')
ax.set_xlabel('X')
ax.set_ylabel('y')
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()
plt.show()


## Step 2 — Build models of increasing complexity

Create a `models` dictionary with four regressors:

- **Dummy (mean)**: always predicts the training mean — high bias, simplest baseline
- **Degree 1**: linear fit — still too simple for curved data
- **Degree 2**: matches the true quadratic form — should generalize well
- **Degree 9**: very flexible — can memorize training noise (high variance)

Use `Pipeline` with `PolynomialFeatures` + `LinearRegression` for the polynomial models.


### Step 2. Define the `models` dictionary

In [ ]:
models = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'Degree 1': Pipeline([
        ('poly', PolynomialFeatures(degree=1)),
        ('regressor', LinearRegression()),
    ]),
    'Degree 2': Pipeline([
        ('poly', PolynomialFeatures(degree=2)),
        ('regressor', LinearRegression()),
    ]),
    'Degree 9': Pipeline([
        ('poly', PolynomialFeatures(degree=9)),
        ('regressor', LinearRegression()),
    ]),
}


## Step 3 — Visualize the fits

Fit each model on the full `(X, y)` and plot the learned curve over a dense grid of `X` values. Use a **2×2** subplot grid — one panel per model.

**Predict:** which degree will hug the training noise most tightly?


### Step 3. Plot fitted curves for all four models

In [ ]:
X_plot = np.linspace(X.min(), X.max(), num=100).reshape(-1, 1)

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(8, 8),
    constrained_layout=True,
)
axes = axes.flatten()

curve_colors = ['green', 'red', 'deeppink', 'orange']

for axis, (name, model), color in zip(axes, models.items(), curve_colors):
    model.fit(X, y)
    y_plot = model.predict(X_plot)

    sns.scatterplot(
        data=data_df,
        x='X',
        y='y',
        color='steelblue',
        alpha=0.6,
        ax=axis,
    )
    axis.plot(X_plot, y_plot, color=color, linewidth=2)
    axis.set_xlim(-3, 3)
    axis.set_ylim(-2, 10)
    axis.grid(True, linestyle='--', alpha=0.5)
    axis.set_title(name)

plt.show()


## Step 4 — Cross-validation on one model

**Cross-validation (CV)** estimates how well a model generalizes **without relying on a single train/test split**. The data is split repeatedly into train/validation folds; each fold scores the model on data it did not train on.

![K-fold cross-validation in parallel](../../assets/cv_parallel.png)

We pass a CV strategy to [`cross_validate`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) via the `cv` parameter.

With `n_jobs=-1`, each fold runs on its own worker:

```{mermaid}
graph LR
  A[Training Data] -->|Split 1| B1[Worker 1]
  A -->|Split 2| B2[Worker 2]
  A -->|Split 3| B3[Worker 3]
  A -->|"..."| B4["..."]
  A -->|Split 10| B10[Worker 10]
  B1 -- Score --> C[Aggregate Scores]
  B2 -- Score --> C
  B3 -- Score --> C
  B4 -- Score --> C
  B10 -- Score --> C
```

**Predict:** will train MAE or test MAE be lower for Degree 1?


### Step 4.a Build `KFold` and cross-validate `models['Degree 1']`

In [ ]:
k_fold = KFold(
    n_splits=10,       # split training data into 10 folds
    shuffle=True,      # randomize row order before splitting
    random_state=42,   # reproducible fold assignments
)

cv_results = cross_validate(
    estimator=models['Degree 1'],
    X=X,
    y=y,
    cv=k_fold,
    n_jobs=-1,  # train and score each fold on a separate CPU core
    scoring='neg_mean_absolute_error',  # sklearn: higher is better, so MAE is negated
    return_train_score=True,
)


### Step 4.b Inspect per-fold MAE

Flip the sign back on `train_score` and `test_score` to get **MAE** (lower is better).


In [ ]:
cv_df = pd.DataFrame(cv_results)
cv_df['test_score'] = -cv_df['test_score']
cv_df['train_score'] = -cv_df['train_score']

cv_df[['train_score', 'test_score']]


## Step 5 — Cross-validate all four models

Loop over every model in `models`, run `cross_validate`, and collect per-fold train and test MAE into a long-format `results_df` with columns `Model`, `Score`, and `Type` (`train` or `test`).


### Step 5. Cross-validate all models and build `results_df`

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=k_fold,
        n_jobs=-1,
        scoring='neg_mean_absolute_error',
        return_train_score=True,
    )

    for train_mae in -scores['train_score']:
        cv_results.append({'Model': name, 'Score': train_mae, 'Type': 'train'})

    for test_mae in -scores['test_score']:
        cv_results.append({'Model': name, 'Score': test_mae, 'Type': 'test'})

results_df = pd.DataFrame(cv_results)
results_df


## Step 6 — Compare by cross-validated MAE

Plot a grouped bar chart of `Score` by `Model`, with `hue='Type'` to separate train vs test folds.

**Predict:** which model has the smallest gap between train and test MAE?


### Step 6. Barplot of cross-validated MAE

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.set_theme(style='whitegrid')

sns.barplot(
    data=results_df,
    x='Model',
    y='Score',
    hue='Type',
    palette='viridis',
    ax=ax,
)

ax.set_title('Models cross-validated on 10 folds each')
ax.set_ylabel('MAE (lower is better)')
ax.set_xlabel('Model')
ax.set_ylim(0, 3)
fig.tight_layout()
plt.show()


## Explore further

Optional stretch goals:

1. Try other polynomial degrees (3, 5, 15) — where does test MAE start climbing?
2. Change `n_samples` or the noise level — how does that affect overfitting?
3. Change `n_splits` in `KFold` — do the scores become more or less stable?


### Explore — your experiments

In [ ]:
# Example: compare degree 3 vs degree 15 on the same data
extra_degrees = [3, 5, 15]
extra_results = []

for degree in extra_degrees:
  model = Pipeline([
      ('poly', PolynomialFeatures(degree=degree)),
      ('regressor', LinearRegression()),
  ])
  scores = cross_validate(
      estimator=model,
      X=X,
      y=y,
      cv=k_fold,
      n_jobs=-1,
      scoring='neg_mean_absolute_error',
      return_train_score=True,
  )
  extra_results.append({
      'degree': degree,
      'mean_train_mae': -scores['train_score'].mean(),
      'mean_test_mae': -scores['test_score'].mean(),
  })

pd.DataFrame(extra_results)


---

## Wrap-up — Reflect

Answer in a markdown cell or your notes:

1. **Curves:** Which model under-fits? Which over-fits? How can you tell from the Step 3 plots?
2. **Bias vs variance:** In your own words, what is the bias–variance tradeoff?
3. **Cross-validation:** Why is CV more reliable than a single train/test split on a dataset this small?
4. **Deployment:** Which model would you ship, and why?

**Key takeaway:** more complexity reduces training error but can hurt generalization. Cross-validated scores help you pick complexity before you touch the final test set.
